In [ ]:
from pathlib import Path
from sahi.slicing import slice_image

# 输入图片路径
image_path = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/06_rpi/predict/0727_0836_640.jpg")

# 切片输出目录
output_dir = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/06_rpi/predict/results/sahi_slices")
output_dir.mkdir(parents=True, exist_ok=True)

# SAHI 切片参数（可按需调整）
slice_height = 640
slice_width = 640
overlap_height_ratio = 0.1
overlap_width_ratio = 0.1

result = slice_image(
    image=str(image_path),
    output_file_name=image_path.stem,
    output_dir=str(output_dir),
    slice_height=slice_height,
    slice_width=slice_width,
    overlap_height_ratio=overlap_height_ratio,
    overlap_width_ratio=overlap_width_ratio,
    verbose=True,
)

print(f"总切片数: {len(result.coco_images)}")
print(f"切片保存目录: {output_dir}")

2026-03-27 13:47:41,973 - sahi - INFO - image.shape: (4656, 3496) (slicing.py:325)
2026-03-27 13:47:42,066 - sahi - INFO - Num slices: 48 slice_height: 640 slice_width: 640 (slicing.py:399)


总切片数: 48
切片保存目录: /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/06_rpi/predict/results/sahi_slices


2026-03-27 13:47:42,088 - sahi - INFO - sliced image path: /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/06_rpi/predict/results/sahi_slices/0727_0836_640_4016_576_4656_1216.png (slicing.py:317)
2026-03-27 13:47:42,094 - sahi - INFO - sliced image path: /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/06_rpi/predict/results/sahi_slices/0727_0836_640_2304_576_2944_1216.png (slicing.py:317)
2026-03-27 13:47:42,100 - sahi - INFO - sliced image path: /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/06_rpi/predict/results/sahi_slices/0727_0836_640_1728_576_2368_1216.png (slicing.py:317)
2026-03-27 13:47:42,104 - sahi - INFO - sliced image path: /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/06_rpi/predict/results/sahi_slices/0727_0836_640_576_576_1216_1216.png (slicing.py:317)
2026-03-27 13:47:42,109 - sahi - INFO - sliced image path: /home/tianqi/D/01_Proj

In [4]:
import cv2
import numpy as np

# 读取原图尺寸，用于创建回拼画布
orig_img = cv2.imread(str(image_path))
if orig_img is None:
    raise FileNotFoundError(f"无法读取原图: {image_path}")

orig_h, orig_w = orig_img.shape[:2]
stitched = np.zeros((orig_h, orig_w, 3), dtype=np.uint8)

# 先把切片贴回原图位置
for tile, (start_x, start_y) in zip(result.images, result.starting_pixels):
    tile_h, tile_w = tile.shape[:2]

    end_x = min(start_x + tile_w, orig_w)
    end_y = min(start_y + tile_h, orig_h)

    valid_w = end_x - start_x
    valid_h = end_y - start_y
    if valid_w <= 0 or valid_h <= 0:
        continue

    stitched[start_y:end_y, start_x:end_x] = tile[:valid_h, :valid_w]

# 分割线样式（BGR）
line_color = (0, 0, 255)  # 红色
line_thickness = 2

# overlap 影响下的实际步长（相邻切片起点间距）
stride_x = max(1, int(round(slice_width * (1 - overlap_width_ratio))))
stride_y = max(1, int(round(slice_height * (1 - overlap_height_ratio))))

# 按切片起点网格画线：这才是考虑 overlap 后的真实切分线
for x in range(stride_x, orig_w, stride_x):
    cv2.line(stitched, (x, 0), (x, orig_h - 1), line_color, line_thickness)

for y in range(stride_y, orig_h, stride_y):
    cv2.line(stitched, (0, y), (orig_w - 1, y), line_color, line_thickness)

# 保存结果
rebuild_path = output_dir / f"{image_path.stem}_stitched_with_grid_overlap.jpg"
cv2.imwrite(str(rebuild_path), stitched)

print(f"拼接完成: {rebuild_path}")
print(f"stride_x={stride_x}, stride_y={stride_y}")

拼接完成: /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/06_rpi/predict/results/sahi_slices/0727_0836_640_stitched_with_grid_overlap.jpg
stride_x=576, stride_y=576
